In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import time

FOLDER = "/content/drive/MyDrive/deep learning"
SRC    = f"{FOLDER}/PSIDSHELF_1968_2021_LONG.dta"

t0 = time.time()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install pyreadstat -q

import pyreadstat

COLS = [
    "ID", "YEAR", "DEMO_BIRTH_YEAR", "DEMO_AGE_GEN", "REFCOUPLE",
    "DEMO_SEX", "RACE_ETH_MAJ_COL", "CGEO_REGION",
    "REL_PAR_BF_ID", "REL_PAR_BM_ID",
    "EARN_TOT_RDF", "FINC_TOT_RDF",
    "WLTH_TOT_NET_RDF", "WLTH_HOME_NET_RDF", "WLTH_SAVI_NET_RDF",
    "WLTH_INVE_NET_RDF", "WLTH_ODEB_NET_RDF",
    "EMP_WORK", "EDU_LEVEL", "EDU_LEVEL_MAX",
    "FAM_PARSTAT", "FAM_MARSTAT", "FAM_SIZE", "FAM_SIZE_CHI",
    "HOME_STAT", "GEO_REGION", "GEO_METRO",
    "OCC_1970C", "OCC_2000C_1M", "OCC_2010C_1M",
]

raw, meta = pyreadstat.read_dta(SRC, usecols=COLS, disable_datetime_conversion=True)
print(f"Loaded: {len(raw):,} person-year rows, {raw.shape[1]} columns")
print(f"Memory: {raw.memory_usage(deep=True).sum() / 1e9:.2f} GB")

Loaded: 3,533,082 person-year rows, 30 columns
Memory: 2.73 GB


In [ ]:
egos = (raw
        .query("REFCOUPLE == 1")
        .query("1960 <= DEMO_BIRTH_YEAR <= 1975")
        [["ID", "DEMO_BIRTH_YEAR", "REL_PAR_BF_ID", "REL_PAR_BM_ID"]]
        .drop_duplicates()
        .copy())
egos["yr_lo"] = egos["DEMO_BIRTH_YEAR"] + 10
egos["yr_hi"] = egos["DEMO_BIRTH_YEAR"] + 18
print(f"{len(egos):,} egos in the 1960-1975 refcouple cohort")

10,948 egos in the 1960-1975 refcouple cohort


In [ ]:
# Father features
f_parent = (raw[["ID", "YEAR", "EARN_TOT_RDF", "FINC_TOT_RDF",
                 "EDU_LEVEL_MAX", "FAM_MARSTAT"]]
            .rename(columns={"ID": "REL_PAR_BF_ID", "YEAR": "parent_YEAR",
                             "EARN_TOT_RDF": "f_EARN", "FINC_TOT_RDF": "f_FINC",
                             "EDU_LEVEL_MAX": "f_EDU", "FAM_MARSTAT": "f_MAR"}))

# joinby: one row per (parent-year, matching ego)
f = f_parent.merge(egos[["ID", "REL_PAR_BF_ID", "yr_lo", "yr_hi"]],
                   on="REL_PAR_BF_ID", how="inner")
f = f.query("yr_lo <= parent_YEAR <= yr_hi").copy()
f["_married"]  = (f["f_MAR"] == 1).astype(float)
f["_divorced"] = f["f_MAR"].isin([2, 3]).astype(float)

fathers = (f.groupby("ID")
           .agg(father_EARN_mean     = ("f_EARN",   "mean"),
                father_FINC_mean     = ("f_FINC",   "mean"),
                father_share_married = ("_married", "mean"),
                father_share_divorc  = ("_divorced","mean"),
                father_EARN_sd       = ("f_EARN",   "std"),
                father_EDU_MAX       = ("f_EDU",    "max"),
                father_obs_cnt       = ("f_EARN",   "count"))
           .reset_index())
print(f"Fathers: {len(fathers):,} rows")

del f_parent, f

Fathers: 3,203 rows


In [ ]:
# Mother features
m_parent = (raw[["ID", "YEAR", "EARN_TOT_RDF", "FINC_TOT_RDF",
                 "EDU_LEVEL_MAX", "FAM_MARSTAT"]]
            .rename(columns={"ID": "REL_PAR_BM_ID", "YEAR": "parent_YEAR",
                             "EARN_TOT_RDF": "m_EARN", "FINC_TOT_RDF": "m_FINC",
                             "EDU_LEVEL_MAX": "m_EDU", "FAM_MARSTAT": "m_MAR"}))
m = m_parent.merge(egos[["ID", "REL_PAR_BM_ID", "yr_lo", "yr_hi"]],
                   on="REL_PAR_BM_ID", how="inner")
m = m.query("yr_lo <= parent_YEAR <= yr_hi").copy()
m["_married"]  = (m["m_MAR"] == 1).astype(float)
m["_divorced"] = m["m_MAR"].isin([2, 3]).astype(float)

mothers = (m.groupby("ID")
           .agg(mother_EARN_mean     = ("m_EARN",   "mean"),
                mother_FINC_mean     = ("m_FINC",   "mean"),
                mother_share_married = ("_married", "mean"),
                mother_share_divorc  = ("_divorced","mean"),
                mother_EARN_sd       = ("m_EARN",   "std"),
                mother_EDU_MAX       = ("m_EDU",    "max"),
                mother_obs_cnt       = ("m_EARN",   "count"))
           .reset_index())
print(f"Mothers: {len(mothers):,} rows")

del m_parent, m

Mothers: 4,347 rows


In [ ]:
# Sample filters
df = raw.query("REFCOUPLE == 1").copy()
print(f"After REFCOUPLE==1: {len(df):,} rows")

df = df.query("1960 <= DEMO_BIRTH_YEAR <= 1975").copy()
print(f"After cohort 1960-1975: {len(df):,} rows")

df = df.query("20 <= DEMO_AGE_GEN <= 45").copy()
print(f"After ages 20-45: {len(df):,} rows")

del raw

After REFCOUPLE==1: 487,555 rows
After cohort 1960-1975: 96,077 rows
After ages 20-45: 77,833 rows


In [ ]:
# Coverage filter: >=8 waves at 20-40 AND >=2 waves at 41-45


# Note for Zachk: we can relax to >= 6 waves or leave it as is?


in_input  = df["DEMO_AGE_GEN"].between(20, 40).astype(int)
in_target = df["DEMO_AGE_GEN"].between(41, 45).astype(int)
df["n_in"] = in_input.groupby(df["ID"]).transform("sum")
df["n_tg"] = in_target.groupby(df["ID"]).transform("sum")
df = df.query("n_in >= 8 & n_tg >= 2").drop(columns=["n_in", "n_tg"]).copy()

print(f"After coverage filter: {len(df):,} rows")
print(f"Unique egos surviving: {df['ID'].nunique():,}")

After coverage filter: 38,286 rows
Unique egos surviving: 2,585


In [ ]:
# Keep export columns + unified occupation
KEEP = [
    "ID", "YEAR", "DEMO_BIRTH_YEAR", "DEMO_AGE_GEN",
    "DEMO_SEX", "RACE_ETH_MAJ_COL", "CGEO_REGION",
    "REL_PAR_BF_ID", "REL_PAR_BM_ID",
    "EARN_TOT_RDF", "FINC_TOT_RDF",
    "WLTH_TOT_NET_RDF", "WLTH_HOME_NET_RDF", "WLTH_SAVI_NET_RDF",
    "WLTH_INVE_NET_RDF", "WLTH_ODEB_NET_RDF",
    "EMP_WORK", "EDU_LEVEL",
    "FAM_PARSTAT", "FAM_MARSTAT", "FAM_SIZE", "FAM_SIZE_CHI",
    "HOME_STAT", "GEO_REGION", "GEO_METRO",
    "OCC_1970C", "OCC_2000C_1M", "OCC_2010C_1M",
]
df = df[KEEP].copy()

# Unified occupation — whichever scheme is populated this row
df["OCC_ANY"] = df["OCC_1970C"]
mask = df["OCC_ANY"].isna() & df["OCC_2000C_1M"].notna()
df.loc[mask, "OCC_ANY"] = df.loc[mask, "OCC_2000C_1M"]
mask = df["OCC_ANY"].isna() & df["OCC_2010C_1M"].notna()
df.loc[mask, "OCC_ANY"] = df.loc[mask, "OCC_2010C_1M"]

def occ_scheme(r):
    if pd.notna(r["OCC_1970C"]):    return 1
    if pd.notna(r["OCC_2000C_1M"]): return 2
    if pd.notna(r["OCC_2010C_1M"]): return 3
    return np.nan
df["OCC_SCHEME"] = df.apply(occ_scheme, axis=1)

print(f"Rows with any occupation: {df['OCC_ANY'].notna().sum():,} / {len(df):,}")


# Note for Zachk: embed the three OCC_* columns separately; the integer codes
# are NOT comparable across the three schemes. OCC_ANY is a convenience only.



Rows with any occupation: 31,691 / 38,286


In [ ]:
# Merge parental features + PAR_BOTH_MISS diagnostic
df = df.merge(fathers, on="ID", how="left")
df = df.merge(mothers, on="ID", how="left")
df["PAR_BOTH_MISS"] = (df["father_EARN_mean"].isna() &
                      df["mother_EARN_mean"].isna()).astype("int8")

# Ego-level diagnostic
ego_diag = df[["ID", "PAR_BOTH_MISS"]].drop_duplicates()
print(ego_diag["PAR_BOTH_MISS"].value_counts().rename(
    {0: "has-parent(s)", 1: "neither parent"}))


# Note for Zachk: 1,711 (66.19%) have at least one parent's info, 874 (33.81%) have neither, we can ask TA, but should be fine

PAR_BOTH_MISS
has-parent(s)     1711
neither parent     874
Name: count, dtype: int64


In [ ]:
# Winsorize (1/99) + signed-log transform
DOLLAR = [
    "EARN_TOT_RDF", "FINC_TOT_RDF",
    "WLTH_TOT_NET_RDF", "WLTH_HOME_NET_RDF", "WLTH_SAVI_NET_RDF",
    "WLTH_INVE_NET_RDF", "WLTH_ODEB_NET_RDF",
    "father_EARN_mean", "father_EARN_sd", "father_FINC_mean",
    "mother_EARN_mean", "mother_EARN_sd", "mother_FINC_mean",
]
for v in DOLLAR:
    s = df[v].dropna()
    lo, hi = s.quantile([0.01, 0.99])
    print(f"{v:30s}  p1 = {lo:12,.0f}    p99 = {hi:12,.0f}")
    df[v] = df[v].clip(lower=lo, upper=hi)
    df[v] = np.sign(df[v]) * np.log1p(df[v].abs() / 1000.0)



EARN_TOT_RDF                    p1 =            0    p99 =      131,526
FINC_TOT_RDF                    p1 =        1,248    p99 =      219,813
WLTH_TOT_NET_RDF                p1 =      -73,268    p99 =    1,223,241
WLTH_HOME_NET_RDF               p1 =      -17,934    p99 =      356,527
WLTH_SAVI_NET_RDF               p1 =            0    p99 =      116,731
WLTH_INVE_NET_RDF               p1 =            0    p99 =      317,303
WLTH_ODEB_NET_RDF               p1 =            0    p99 =      100,009
father_EARN_mean                p1 =            0    p99 =      147,602
father_EARN_sd                  p1 =            0    p99 =       60,156
father_FINC_mean                p1 =        7,214    p99 =      554,725
mother_EARN_mean                p1 =            0    p99 =       37,225
mother_EARN_sd                  p1 =            0    p99 =       16,066
mother_FINC_mean                p1 =        4,666    p99 =      180,159


In [ ]:
# Birth cohort bin
cond = [df["DEMO_BIRTH_YEAR"].between(1960, 1964),
        df["DEMO_BIRTH_YEAR"].between(1965, 1969),
        df["DEMO_BIRTH_YEAR"].between(1970, 1975)]
df["COHORT_BIN"] = np.select(cond, [0, 1, 2], default=np.nan).astype("int8")

In [ ]:
# Random 50/25/25 split on ID (seed 1470 for reproducibility)
rng = np.random.default_rng(1470)
ids = np.sort(df["ID"].unique())
u = rng.uniform(size=len(ids))
split_lab = np.where(u < 0.50, "train",
             np.where(u < 0.75, "val", "test"))
splits = pd.DataFrame({"ID": ids, "split": split_lab})

print(f"Total egos in splits: {len(splits):,}")
print(splits["split"].value_counts())


df = df.merge(splits, on="ID", how="left")


# Note for Zachk: with this data size, we can try to avoid overfitting in LSTM



Total egos in splits: 2,585
split
train    1282
test      654
val       649
Name: count, dtype: int64


In [ ]:
df = df.sort_values(["ID", "DEMO_AGE_GEN", "YEAR"]).reset_index(drop=True)
print(f"Final rows to export: {len(df):,}")
print(df.dtypes)

df.to_csv(f"{FOLDER}/psid_panel.csv", index=False)
splits.to_csv(f"{FOLDER}/splits.csv", index=False)

print(f"DONE in {time.time() - t0:.1f}s")
print(f"Written: {FOLDER}/psid_panel.csv, {FOLDER}/splits.csv")

Final rows to export: 38,286
ID                        int64
YEAR                      int64
DEMO_BIRTH_YEAR          object
DEMO_AGE_GEN             object
DEMO_SEX                 object
RACE_ETH_MAJ_COL         object
CGEO_REGION              object
REL_PAR_BF_ID            object
REL_PAR_BM_ID            object
EARN_TOT_RDF            float64
FINC_TOT_RDF            float64
WLTH_TOT_NET_RDF        float64
WLTH_HOME_NET_RDF       float64
WLTH_SAVI_NET_RDF       float64
WLTH_INVE_NET_RDF       float64
WLTH_ODEB_NET_RDF       float64
EMP_WORK                 object
EDU_LEVEL                object
FAM_PARSTAT              object
FAM_MARSTAT              object
FAM_SIZE                 object
FAM_SIZE_CHI             object
HOME_STAT                object
GEO_REGION               object
GEO_METRO                object
OCC_1970C                object
OCC_2000C_1M             object
OCC_2010C_1M             object
OCC_ANY                  object
OCC_SCHEME              float64
father_EARN

In [ ]:
#-----------------------------------------------------Data Cleaning V1 Done-----------------------------------------------